# Advanced GEAP managed evaluation

This notebook is a **thin demo**: every step calls into the tested `geap_tuning`
package. It mirrors [`examples/run_advanced_eval.py`](../examples/run_advanced_eval.py).

GEAP's managed **Evaluation** service runs *inside* a tuning job — there is no
standalone eval client in google-genai 2.14.0. You attach an `EvaluationConfig`
to the tuning job and GEAP evaluates each checkpoint, writing results to Cloud
Storage. This demo attaches a **comprehensive** config over the SFT support-intent
dataset ([SFT notebook](01_sft.ipynb)):

| Piece | Builder | Role |
|---|---|---|
| LLM-judge metric | `llm_judge_metric` | a judge model scores each response |
| computation metrics | `computation_metric` | deterministic `EXACT_MATCH` / `ROUGE` |
| predefined metric | `predefined_metric` | managed catalog metric by name |
| autorater config | `build_autorater_config` | tunes the shared judge model |
| inference config | `types.GenerationConfig` | how the tuned model generates responses |

Eval runs per exported checkpoint, so cadence follows checkpointing. There is
**no** `evaluate_interval` knob for SFT: in google-genai 2.14.0 that field
serializes only under the reinforcement spec, so passing it on an SFT job 400s —
it is RLFT-only (see [notebook 06](06_rlft_reward_types.ipynb)).

> **Constraints:** managed eval is **Preview, `us-central1`-only**; the SDK
> **lowercases** `Metric.name`; predefined metric names must exist in the live
> catalog — verify before running.

> **Requires live GCP and incurs tuning cost.** Have a real `.env` and
> `gcloud auth` in place before the tune cell.

In [ ]:
from pathlib import Path

from google.genai import types

from geap_tuning.config import genai_client, load_config

VERSION = "v1"
DISPLAY_NAME = f"geap-eval-{VERSION}"
DATA_DIR = Path("datasets/sft_support_intent")
GCS_PREFIX = "sft_support_intent"
EVAL_PREFIX = "advanced_eval"
EVAL_REGION = "us-central1"  # managed eval is us-central1-only (Preview)

cfg = load_config()
client = genai_client(cfg)
if cfg.location != EVAL_REGION:
    print(f"WARNING: managed eval is {EVAL_REGION}-only; cfg.location={cfg.location}")
cfg

## Assemble the comprehensive eval config

Mix all three metric kinds, then set the shared autorater config and a
deterministic inference config (`temperature=0.0`) so scores are comparable across
checkpoints. `build_evaluation_config` writes results under `bucket/EVAL_PREFIX`.

In [ ]:
from geap_tuning.autoeval import (
    build_autorater_config,
    build_evaluation_config,
    computation_metric,
    llm_judge_metric,
    predefined_metric,
)

metrics = [
    llm_judge_metric(
        "intent_correctness",
        "Does the response state the correct support intent for the ticket? "
        "Respond with a score from 1 (wrong) to 5 (exactly right).\n{prediction}",
    ),
    computation_metric(types.ComputationBasedMetricType.EXACT_MATCH),
    computation_metric(types.ComputationBasedMetricType.ROUGE),
    predefined_metric("text_quality_v1"),
]
eval_config = build_evaluation_config(
    cfg.bucket,
    prefix=EVAL_PREFIX,
    metrics=metrics,
    autorater_config=build_autorater_config(sampling_count=4),
    inference_generation_config=types.GenerationConfig(temperature=0.0),
)
len(eval_config.metrics)

## Stage data + launch with eval attached

`export_last_checkpoint_only=False` (default) keeps intermediate checkpoints so
eval runs per checkpoint. Reuse an existing job by display name for idempotent
reruns.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.jobs import find_tuning_job_by_display_name, wait_for_tuning_job
from geap_tuning.sft.data import build_sft_dataset
from geap_tuning.sft.tune import launch_sft_job

paths = build_sft_dataset(DATA_DIR)
train_uri = upload_file(paths["train"], f"{cfg.bucket}/{GCS_PREFIX}/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/{GCS_PREFIX}/val.jsonl")

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_sft_job(
        client,
        train_uri=train_uri,
        val_uri=val_uri,
        display_name=DISPLAY_NAME,
        evaluation_config=eval_config,
        labels=cfg.labels,
    )
job.name

## Wait + locate results

Eval results land as JSON under the configured GCS prefix, one set per evaluated
checkpoint.

In [ ]:
job = wait_for_tuning_job(client, job.name)
prefix = eval_config.output_config.gcs_destination.output_uri_prefix
print(f"Eval results under: {prefix}")
print(f"Inspect with: gcloud storage ls {prefix}")